# 04 — Fraud labeling (Part B 2019–2023)

Replicates Johnson & Khoshgoftaar (2023), "Fraud labeling" section:
1. Get the **active LEIE list as of January of each data year** from the Wayback Machine.
2. Keep only the **7 fraud-related exclusion rules** (Table 6).
3. **Concatenate** all snapshots. For each NPI, use the **most recent record**.
4. **Last exclusion year** = exclusion year + minimum exclusion period, rounded to the nearest year.
5. Rows with `year` **before** the last exclusion year → `fraud = 1`. Everything else → `fraud = 0`.

Input: `processed/part_b/part_b_enriched_2019_2023.parquet` (5,626,423 × 90)
Output: `processed/part_b/labeled/part_b_labeled_2019_2023.parquet` (same rows + `fraud`)

In [29]:
# Imports and folder paths. DATA_DIR falls back to the scratch root if the env var isn't set.
import os, json, time, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR  = Path(os.environ.get("DATA_DIR", "/scratch/sm3664/medicare-fraud"))
LEIE_DIR  = DATA_DIR / "raw" / "leie" / "wayback"
PROC_DIR  = DATA_DIR / "processed" / "part_b"
OUT_DIR   = PROC_DIR / "labeled"
LEIE_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

ENRICHED = PROC_DIR / "part_b_enriched_2019_2023.parquet"
print(DATA_DIR, ENRICHED.exists())

/scratch/sm3664/medicare-fraud True


## Settings

- `SNAPSHOT_YEARS`: the paper took one January snapshot per data year (2013–2019). We mirror that with 2019–2023.
- `FRAUD_RULES`: Table 6 of the paper.
- `MIN_PERIOD_YEARS`: minimum exclusion period per rule, from SSA §1128(c)(3):
  - (a)(1)–(a)(3): 5 years
  - 2nd offense (c)(3)(G)(i): 10 years
  - 3rd offense (c)(3)(G)(ii): permanent
  - ⚠️ (b)(4) and (b)(7) have **no fixed statutory minimum**. (b)(4) is tied to the license action; (b)(7) is set case by case. The paper doesn't say what it used. We use 0 (the exclusion year itself) as an assumption to document.
- ⚠️ `LABEL_RULE`: the paper says "prior to" the last exclusion year, so we use `<`. Switching to `<=` is a sensitivity check.
- ⚠️ `ADD_CURRENT_LIST`: the paper uses only January snapshots, so providers excluded during 2023 after January are missed. Default is `False`, which is the paper's method.

In [30]:
# All choices that affect labels live here, so they're visible and easy to change.
SNAPSHOT_YEARS = [2019, 2020, 2021, 2022, 2023]
LEIE_URL = "https://oig.hhs.gov/exclusions/downloadables/UPDATED.csv"

FRAUD_RULES = ["1128a1", "1128a2", "1128a3", "1128b4", "1128b7", "1128c3gi", "1128c3gii"]

PERMANENT = 9999
MIN_PERIOD_YEARS = {
    "1128a1": 5, "1128a2": 5, "1128a3": 5,
    "1128c3gi": 10,
    "1128c3gii": PERMANENT,
    "1128b4": 0,   # ASSUMPTION: no statutory minimum
    "1128b7": 0,   # ASSUMPTION: no statutory minimum
}

LABEL_RULE = "<"          # "<" = paper wording ("prior to"); "<=" for sensitivity check
ADD_CURRENT_LIST = True   # paper: current LEIE + historical January snapshots  # True = also add today's LEIE list (not in paper)

## Step 1a — Find the January snapshots

This asks the Wayback **CDX API** for captures of `UPDATED.csv` in each year's January.
- If January has none, the window widens to **Feb–Mar**. Last time, 2018 needed this.
- Only HTTP 200 captures count. The earliest one in the window is used.
- If the internet check fails, compute nodes can't reach the web. Run Steps 1a–1b from the login node instead.

In [31]:
# Find one Wayback capture per year. Prints the chosen timestamp so you can verify it.
UA = {"User-Agent": "Mozilla/5.0 (research; Rutgers)"}

def get(url, timeout=60):
    req = urllib.request.Request(url, headers=UA)
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

try:
    get("https://web.archive.org", timeout=15); print("Internet OK")
except Exception as e:
    raise SystemExit(f"No internet from this node: {e}")

def find_capture(year):
    for start, end in [(f"{year}0101", f"{year}0131"), (f"{year}0101", f"{year}0331")]:
        q = (f"https://web.archive.org/cdx/search/cdx?url={LEIE_URL}&output=json"
             f"&from={start}&to={end}&filter=statuscode:200&fl=timestamp,original")
        rows = json.loads(get(q) or b"[]")
        if len(rows) > 1:                     # first row is the header
            return rows[1][0], rows[1][1], (start, end)
    return None, None, None

captures = {}
for y in SNAPSHOT_YEARS:
    ts, orig, win = find_capture(y)
    captures[y] = (ts, orig)
    print(y, ts, "window:", win)
    time.sleep(1)                             # be polite to the API

Internet OK
2019 20190110211040 window: ('20190101', '20190131')
2020 20200310231828 window: ('20200101', '20200331')
2021 20210318000849 window: ('20210101', '20210331')
2022 20220311002415 window: ('20220101', '20220331')
2023 20230110203218 window: ('20230101', '20230131')


In [32]:
# Save each snapshot as UPDATED_<timestamp>.csv and validate it.
def looks_like_leie(data):
    head = data[:200].lstrip(b"\xef\xbb\xbf").lstrip().replace(b'"', b"").upper()
    return head.startswith(b"LASTNAME,FIRSTNAME")

snap_files = {}
for y, (ts, orig) in captures.items():
    if ts is None:
        print(f"{y}: NO CAPTURE — decide how to handle before labeling"); continue
    f = LEIE_DIR / f"UPDATED_{ts}.csv"
    if not f.exists():
        data = get(f"https://web.archive.org/web/{ts}id_/{orig}", timeout=300)
        assert looks_like_leie(data), f"{y}: not a CSV (got {data[:80]!r})"
        f.write_bytes(data); time.sleep(2)
    snap_files[y] = f
    print(y, f.name, f"{f.stat().st_size/1e6:.1f} MB")

2019 UPDATED_20190110211040.csv 12.8 MB
2020 UPDATED_20200310231828.csv 13.1 MB
2021 UPDATED_20210318000849.csv 13.4 MB
2022 UPDATED_20220311002415.csv 13.6 MB
2023 UPDATED_20230110203218.csv 13.9 MB


In [33]:
# Load each snapshot and record which January list it came from.
def load_leie(path, snap_year):
    d = pd.read_csv(path, dtype=str, keep_default_na=False)
    d.columns = d.columns.str.strip().str.upper()
    d["EXCLTYPE"] = d["EXCLTYPE"].str.lower().str.replace(r"[^0-9a-z]", "", regex=True)
    d["NPI"] = pd.to_numeric(d["NPI"].str.strip(), errors="coerce").fillna(0).astype("int64")
    d["EXCLDATE"] = pd.to_datetime(d["EXCLDATE"].str.strip(), format="%Y%m%d", errors="coerce")
    d["snapshot_year"] = snap_year
    return d[["NPI", "EXCLTYPE", "EXCLDATE", "snapshot_year"]]

frames = [load_leie(f, y) for y, f in snap_files.items()]
if ADD_CURRENT_LIST:
    cur = LEIE_DIR.parent / "UPDATED.csv"
    frames.append(load_leie(cur, 9999))

leie_all = pd.concat(frames, ignore_index=True)

summary = leie_all.groupby("snapshot_year").agg(
    rows=("NPI", "size"),
    fraud_rule_rows=("EXCLTYPE", lambda s: s.isin(FRAUD_RULES).sum()),
    usable_npi=("NPI", lambda s: (s > 0).sum()),
)
print(summary)
print("Unrecognized EXCLTYPE values:", sorted(set(leie_all.EXCLTYPE) - set(MIN_PERIOD_YEARS) - {""})[:30])

                rows  fraud_rule_rows  usable_npi
snapshot_year                                    
2019           70973            62192        4981
2020           73039            64043        5427
2021           74386            65309        5794
2022           75445            66268        6149
2023           77165            67832        6581
9999           84093            74284        8924
Unrecognized EXCLTYPE values: ['1128a4', '1128aa', '1128b1', '1128b11', '1128b12', '1128b14', '1128b15', '1128b16', '1128b2', '1128b3', '1128b5', '1128b6', '1128b8', '1156', '1160', 'brchcia', 'brchsa']


In [34]:
fraud = leie_all[leie_all.EXCLTYPE.isin(FRAUD_RULES) & (leie_all.NPI > 0)]
bad_dates = fraud.EXCLDATE.isna().sum()
fraud = fraud.dropna(subset=["EXCLDATE"])
print("Dropped for bad EXCLDATE:", bad_dates)

cum = (fraud.sort_values(["NPI", "EXCLDATE", "snapshot_year"])
            .drop_duplicates("NPI", keep="last")
            .reset_index(drop=True))

print("Unique fraud NPIs (cumulative):", cum.NPI.nunique())
print("…in the latest snapshot alone:", fraud.loc[fraud.snapshot_year == max(snap_files), "NPI"].nunique())
print(cum.EXCLTYPE.value_counts())

Dropped for bad EXCLDATE: 0
Unique fraud NPIs (cumulative): 7970
…in the latest snapshot alone: 5233
EXCLTYPE
1128a1    3767
1128b4    2805
1128a3     756
1128a2     424
1128b7     218
Name: count, dtype: int64


In [35]:
def last_excl_year(row):
    p = MIN_PERIOD_YEARS[row.EXCLTYPE]
    if p == PERMANENT:
        return PERMANENT
    end = row.EXCLDATE + pd.DateOffset(years=p)
    return end.year + (1 if end.month >= 7 else 0)

cum["last_excl_year"] = cum.apply(last_excl_year, axis=1).astype("int64")
print(cum.groupby("EXCLTYPE").last_excl_year.describe()[["min", "50%", "max"]])
cum.to_parquet(OUT_DIR / "leie_cumulative_2019_2023.parquet", index=False)

             min     50%     max
EXCLTYPE                        
1128a1    1994.0  2024.0  2032.0
1128a2    2009.0  2024.0  2032.0
1128a3    2011.0  2025.0  2032.0
1128b4    2000.0  2018.0  2027.0
1128b7    2009.0  2019.0  2027.0


In [36]:
df = pd.read_parquet(ENRICHED)
n0 = len(df)
df["Rndrng_NPI"] = df["Rndrng_NPI"].astype("int64")

df = df.merge(cum[["NPI", "last_excl_year"]], how="left",
              left_on="Rndrng_NPI", right_on="NPI").drop(columns="NPI")
assert len(df) == n0, "Row count changed — duplicate NPIs in LEIE table?"

cmp = df["year"] < df["last_excl_year"] if LABEL_RULE == "<" else df["year"] <= df["last_excl_year"]
df["fraud"] = cmp.fillna(False).astype("int8")

print("NPIs matched in Part B:", df.loc[df.last_excl_year.notna(), "Rndrng_NPI"].nunique())
print("Fraud NPIs after year rule:", df.loc[df.fraud == 1, "Rndrng_NPI"].nunique())

NPIs matched in Part B: 960
Fraud NPIs after year rule: 930


In [37]:
feat_cols = [c for c in df.columns if c not in ("Rndrng_NPI", "year", "last_excl_year", "fraud")]
pos = int(df.fraud.sum())
print(pd.DataFrame({"Records": [len(df)], "Positive count": [pos],
                    "Positive ratio %": [round(100 * pos / len(df), 4)],
                    "Feature count": [len(feat_cols)]}, index=["Part B AE"]))

print(df.groupby("year").fraud.agg(rows="size", fraud="sum", rate="mean"))

ACTIVE_ROWS, ACTIVE_NPIS = 2557, 900
print(f"Cumulative vs active: rows {pos} vs {ACTIVE_ROWS} "
      f"({100*(pos/ACTIVE_ROWS-1):+.1f}%), NPIs {df.loc[df.fraud==1,'Rndrng_NPI'].nunique()} vs {ACTIVE_NPIS}")

           Records  Positive count  Positive ratio %  Feature count
Part B AE  5626423            2652            0.0471             88
         rows  fraud      rate
year                          
2019  1093367    840  0.000768
2020  1085313    671  0.000618
2021  1123589    529  0.000471
2022  1148873    369  0.000321
2023  1175281    243  0.000207
Cumulative vs active: rows 2652 vs 2557 (+3.7%), NPIs 930 vs 900


In [38]:
out = OUT_DIR / "part_b_labeled_2019_2023.parquet"
df.drop(columns="last_excl_year").to_parquet(out, index=False)
print(out, df.shape)

/scratch/sm3664/medicare-fraud/processed/part_b/labeled/part_b_labeled_2019_2023.parquet (5626423, 92)


In [39]:
cur = LEIE_DIR.parent / "UPDATED.csv"
print(f"{cur.stat().st_size/1e6:.1f} MB", sum(1 for _ in open(cur)) - 1, "rows")
print(open(cur).readline()[:80])            # quoted header or not?
chk = load_leie(cur, 9999)
print("Latest EXCLDATE:", chk.EXCLDATE.max())   # ≈ when this list was downloaded

10.8 MB 84093 rows
LASTNAME,FIRSTNAME,MIDNAME,BUSNAME,GENERAL,SPECIALTY,UPIN,NPI,DOB,ADDRESS,CITY,S
Latest EXCLDATE: 2026-10-20 00:00:00


In [40]:
print("Current list alone:", fraud.loc[fraud.snapshot_year == 9999, "NPI"].nunique())

Current list alone: 7270
